
# Step 0 — Precision Agriculture Patent Retrieval

## 1. Objective and scope

This notebook develops a reproducible **patent retrieval
strategy for the Precision Agriculture** sector.

The search strategy is primarily informed by:
- Tey et al. (2024), *Evolution of precision agricultural
technologies: a patent network analysis*.

- Avola et al. (2024), *Precision agriculture and patented innovation: State of the art and
current trends* 

- Rejeb et al. (2024), *Precision agriculture: A bibliometric analysis and research agenda*

**Study period:** 2000–2022

**Unit of analysis:** DOCDB patent family


## Search strategy — three routes

- **R1:** specific CPC/IPC of Precision Agriculture
- **R2:** explicit Precision Agriculture terminology; 
- **R3:** agricultural CPC/IPC AND enabling technology keyword AND PA function keyword.


**Excluded classifications:** CPCs related to livestock and chemical field. **Topical exclusions:** title-based terms for forestry, golf/turf, livestock and ornamental gardening; 


In [1]:

from pathlib import Path
import re

import pandas as pd
import yaml

from sqlalchemy import and_, or_, func, false, case

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS209_APPLN_IPC,
    TLS224_APPLN_CPC,
)

from IPython.display import display


In [2]:

# Locate the project from its root or notebooks folder
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "config" / "sector_PA_FINAL.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config" / "sector_PA_FINAL.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}"
    )

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

# PATSTAT settings
ENV = config["patstat"]["environment"]
IPR_TYPE = config["patstat"]["ipr_type"]

# Temporal scope
YEAR_MIN = int(config["analysis"]["year_min"])
YEAR_MAX = int(config["analysis"]["year_max"])

FAMILY_TIME_COLUMN = config["analysis"]["family_time_column"]

# Keywords
SPECIFIC_PHRASES = config["keywords"]["specific"]
AGRICULTURAL_TERMS = config["keywords"]["agricultural"]
ENABLING_TERMS = config["keywords"]["enabling_technologies"]
PA_FUNCTION_TERMS = config["keywords"]["pa_functions"]
EXCLUSION_TERMS = config["keywords"].get("exclusion_terms", [])

# Patent classifications
SPECIFIC_CPC = config["cpc"]["specific"]
AGRICULTURAL_CPC = config["cpc"]["agricultural_context"]
SPECIFIC_IPC = config["ipc"].get("specific", [])
AGRICULTURAL_IPC = config["ipc"]["agricultural_context"]

EXCLUDE_CPC = config["cpc"].get("exclude", [])
EXCLUDE_IPC = config["ipc"].get("exclude", [])

# Validation
SAMPLE_SIZE = int(config["validation"]["sample_size"])
RANDOM_SEED = int(config["validation"]["random_seed"])

if YEAR_MIN > YEAR_MAX:
    raise ValueError("Invalid year range.")

if FAMILY_TIME_COLUMN != "earliest_filing_year":
    raise ValueError("Expected earliest_filing_year.")

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "step0"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:", CONFIG_PATH)
print("Environment:", ENV)
print("Period:", YEAR_MIN, "-", YEAR_MAX)
print("Output directory:", OUTPUT_DIR)


Configuration: /home/jovyan/team-38-PA/config/sector_PA_FINAL.yaml
Environment: PROD
Period: 2000 - 2022
Output directory: /home/jovyan/team-38-PA/notebooks/outputs/step0



## 4. Keyword and classification matching

Keyword expressions are converted into case-insensitive
regular expressions.

The asterisk (*) is interpreted as a word-ending wildcard,
while spaces, hyphens, and underscores are accepted as
alternative separators.

Keywords are searched in the available patent titles
and abstracts.

For computational efficiency, the terms belonging to
the same keyword group are combined into a single
regular expression.

CPC and IPC classifications are matched using
normalized code prefixes.


In [3]:

def term_to_regex(term):
    """Convert a YAML keyword into a word-bounded regex."""
    term = term.strip().lower()

    if not term:
        raise ValueError("Search terms cannot be empty.")

    parts = re.split(r"[\s\-_]+", term)

    patterns = [
        re.escape(part).replace(
            r"\*", r"[a-zA-Z0-9]*"
        )
        for part in parts
        if part
    ]

    return (
        r"(?:\b|_)"
        + r"[\s\-_/]+".join(patterns)
        + r"(?:\b|_)"
    )


def keyword_condition(patterns):
    """Match any keyword in the combined title and abstract."""
    if not patterns:
        return false()

    doc_text = func.lower(
        func.concat(
            func.coalesce(
                TLS202_APPLN_TITLE.appln_title, ""
            ),
            " ",
            func.coalesce(
                TLS203_APPLN_ABSTR.appln_abstract, ""
            ),
        )
    )

    combined_pattern = "|".join(
        f"(?:{pattern})"
        for pattern in patterns.values()
    )

    return func.regexp_contains(
        doc_text,
        combined_pattern
    )


def symbol_prefix_condition(column, prefixes):
    """Match CPC/IPC prefixes after removing spaces."""
    if not prefixes:
        return false()

    symbol = func.upper(
        func.replace(column, " ", "")
    )

    return or_(*[
        symbol.like(
            prefix.replace(" ", "").upper() + "%"
        )
        for prefix in prefixes
    ])


SPECIFIC_PATTERNS = {
    term: term_to_regex(term)
    for term in SPECIFIC_PHRASES
}

AGRICULTURAL_PATTERNS = {
    term: term_to_regex(term)
    for term in AGRICULTURAL_TERMS
}

ENABLING_PATTERNS = {
    term: term_to_regex(term) for term in ENABLING_TERMS
}

PA_FUNCTION_PATTERNS = {
    term: term_to_regex(term) for term in PA_FUNCTION_TERMS
}

EXCLUSION_PATTERNS = {
    term: term_to_regex(term) for term in EXCLUSION_TERMS
}

print("Specific phrases:", len(SPECIFIC_PATTERNS))
print("Enabling technologies:", len(ENABLING_PATTERNS))
print("PA functions:", len(PA_FUNCTION_PATTERNS))
print("Topical exclusions:", len(EXCLUSION_PATTERNS))

Specific phrases: 7
Enabling technologies: 51
PA functions: 46
Topical exclusions: 23


In [4]:

patstat = PatstatClient(env=ENV)
db = patstat.orm()

print(f"PATSTAT client initialized: {ENV}")


PATSTAT client initialized: PROD




## 6. Define the family population and representative applications

The unit of analysis is the DOCDB patent family.

Families are restricted to the 2000–2022 period using the minimum
`earliest_filing_year` across their applications.

For each eligible family, we retain only the earliest-filed
application with a non-empty English abstract.

All keyword, CPC/IPC and exclusion criteria are evaluated
exclusively on this representative application.

Families without an eligible English abstract are excluded.



In [5]:

family_year = func.min(
    TLS201_APPLN.earliest_filing_year
)

family_scope = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        family_year.label("family_year")
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE,
        TLS201_APPLN.docdb_family_id > 0
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .having(
        family_year.between(YEAR_MIN, YEAR_MAX)
    )
    .subquery()
)

# Select the earliest-filed application with an English abstract
# for each DOCDB family.

english_abstract_available = and_(
    TLS203_APPLN_ABSTR.appln_abstract_lg == "en",
    func.length(
        func.trim(
            func.coalesce(
                TLS203_APPLN_ABSTR.appln_abstract, ""
            )
        )
    ) > 0
)

ranked_english_apps = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        TLS201_APPLN.appln_id.label("appln_id"),
        TLS201_APPLN.appln_filing_date.label("filing_date"),

        func.row_number().over(
            partition_by=TLS201_APPLN.docdb_family_id,
            order_by=[
                TLS201_APPLN.appln_filing_date.asc(),
                TLS201_APPLN.appln_id.asc()
            ]
        ).label("rn")
    )
    .select_from(TLS201_APPLN)
    .join(
        family_scope,
        TLS201_APPLN.docdb_family_id == family_scope.c.fam_id
    )
    .join(
        TLS203_APPLN_ABSTR,
        and_(
            TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
            english_abstract_available
        )
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE,
        TLS201_APPLN.appln_filing_date.isnot(None)
    )
    .subquery()
)

representative_apps = (
    db.query(
        ranked_english_apps.c.fam_id,
        ranked_english_apps.c.appln_id,
        ranked_english_apps.c.filing_date
    )
    .filter(ranked_english_apps.c.rn == 1)
    .subquery()
)

print("Representative applications prepared.")




Representative applications prepared.



## 7. Collect representative-application keyword evidence

Keyword evidence is extracted exclusively from the representative application selected for each DOCDB patent family.

The representative application is the earliest-filed application with a non-empty English abstract.

Three Boolean indicators are generated:

1. **Specific PA terminology (`specific_phrase`):** expressions explicitly identifying Precision Agriculture.
2. **Enabling technologies (`enabling_technology`):** technological solutions such as sensors, GPS/GNSS, UAVs, AI, robotics, and control systems.
3. **PA functions (`pa_function`):** agricultural precision functions such as crop monitoring, yield mapping, variable-rate application, and automated guidance.

Each indicator is true if the title or abstract of the selected application contains at least one matching term.

Keyword evidence from other applications belonging to the same DOCDB family is not considered.

The three indicators are computed together and subsequently combined with CPC/IPC evidence to evaluate the three retrieval routes.



In [6]:

keyword_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),

        func.logical_or(
            keyword_condition(SPECIFIC_PATTERNS)
        ).label("specific_phrase"),

        func.logical_or(
            keyword_condition(ENABLING_PATTERNS)
        ).label("enabling_technology"),

        func.logical_or(
            keyword_condition(PA_FUNCTION_PATTERNS)
        ).label("pa_function"),
    )
    .select_from(TLS201_APPLN)
    .join(
    representative_apps,
    TLS201_APPLN.appln_id == representative_apps.c.appln_id
)
    .outerjoin(
    TLS202_APPLN_TITLE,
    and_(
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
        TLS202_APPLN_TITLE.appln_title_lg == "en"
    )
)
    .join(
    TLS203_APPLN_ABSTR,
    and_(
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
        TLS203_APPLN_ABSTR.appln_abstract_lg == "en"
    )
)
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .subquery()
)

print("Family-level keyword flags prepared.")


Family-level keyword flags prepared.




## 8. Collect representative-application CPC and IPC evidence

CPC and IPC classification evidence is extracted exclusively
from the representative application selected for each
DOCDB patent family.

For the selected application, three Boolean indicators
are generated:

1. Specific Precision Agriculture CPC codes.
2. Agricultural-context CPC codes.
4. Agricultural-context IPC codes.



In [7]:

cpc_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),

        func.logical_or(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                SPECIFIC_CPC
            )
        ).label("specific_cpc"),

        func.logical_or(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                AGRICULTURAL_CPC
            )
        ).label("agricultural_cpc"),
    )
    .select_from(TLS201_APPLN)
    .join(
    representative_apps,
    TLS201_APPLN.appln_id == representative_apps.c.appln_id
)
    .join(
        TLS224_APPLN_CPC,
        TLS201_APPLN.appln_id
        == TLS224_APPLN_CPC.appln_id
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .subquery()
)

print("CPC flags prepared.")


CPC flags prepared.


In [8]:

ipc_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),

        func.logical_or(
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                SPECIFIC_IPC
            )
        ).label("specific_ipc"),

        func.logical_or(
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                AGRICULTURAL_IPC
            )
        ).label("agricultural_ipc"),
    )
    .select_from(TLS201_APPLN)
    .join(
    representative_apps,
    TLS201_APPLN.appln_id == representative_apps.c.appln_id
)
    .join(
        TLS209_APPLN_IPC,
        TLS201_APPLN.appln_id
        == TLS209_APPLN_IPC.appln_id
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .subquery()
)

print("IPC flags prepared.")


IPC flags prepared.


In [9]:

excluded_queries = []

# CPC classification exclusions — representative application only
if EXCLUDE_CPC:
    excluded_queries.append(
        db.query(
            representative_apps.c.fam_id.label("fam_id")
        )
        .select_from(representative_apps)
        .join(
            TLS224_APPLN_CPC,
            representative_apps.c.appln_id
            == TLS224_APPLN_CPC.appln_id
        )
        .filter(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                EXCLUDE_CPC
            )
        )
    )

# IPC classification exclusions — representative application only
if EXCLUDE_IPC:
    excluded_queries.append(
        db.query(
            representative_apps.c.fam_id.label("fam_id")
        )
        .select_from(representative_apps)
        .join(
            TLS209_APPLN_IPC,
            representative_apps.c.appln_id
            == TLS209_APPLN_IPC.appln_id
        )
        .filter(
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                EXCLUDE_IPC
            )
        )
    )

if excluded_queries:
    excluded_query = excluded_queries[0]

    for query in excluded_queries[1:]:
        excluded_query = excluded_query.union(query)

    excluded_families = excluded_query.distinct().subquery()
else:
    excluded_families = None

print("Representative-application classification exclusions prepared.")


Representative-application classification exclusions prepared.


In [10]:

# Title-only thematic exclusions on representative applications

if EXCLUSION_PATTERNS:
    title_exclusion_pattern = "|".join(
        f"(?:{pattern})"
        for pattern in EXCLUSION_PATTERNS.values()
    )

    topical_exclusions = (
        db.query(
            representative_apps.c.fam_id.label("fam_id")
        )
        .select_from(representative_apps)
        .join(
            TLS202_APPLN_TITLE,
            and_(
                representative_apps.c.appln_id
                == TLS202_APPLN_TITLE.appln_id,
                TLS202_APPLN_TITLE.appln_title_lg == "en"
            )
        )
        .filter(
            func.regexp_contains(
                func.lower(
                    func.coalesce(
                        TLS202_APPLN_TITLE.appln_title, ""
                    )
                ),
                title_exclusion_pattern
            )
        )
        .distinct()
        .subquery()
    )
else:
    topical_exclusions = None


# Protected CPC: A01B79/005
# Preserve the agreed protection rule,
# using the representative application only.

protected_families = (
    db.query(
        representative_apps.c.fam_id.label("fam_id")
    )
    .select_from(representative_apps)
    .join(
        TLS224_APPLN_CPC,
        representative_apps.c.appln_id
        == TLS224_APPLN_CPC.appln_id
    )
    .filter(
        symbol_prefix_condition(
            TLS224_APPLN_CPC.cpc_class_symbol,
            ["A01B79/005"]
        )
    )
    .distinct()
    .subquery()
)

print("Representative-application topical exclusions and protected CPC prepared.")


Representative-application topical exclusions and protected CPC prepared.




## 10. Combine representative-application evidence

Keyword and classification indicators are evaluated on the
same representative application, selected for each DOCDB family.

The indicators are then combined at the DOCDB family level
for counting and retrieval.

Missing indicators are interpreted as false.

The search applies the three routes R1, R2 and R3 without
combining evidence from different applications.



In [11]:

flags = (
    db.query(
        family_scope.c.fam_id,
        family_scope.c.family_year,

        func.coalesce(
            keyword_flags.c.specific_phrase, False
        ).label("specific_phrase"),

        func.coalesce(
            keyword_flags.c.enabling_technology, False
        ).label("enabling_technology"),

        func.coalesce(
            keyword_flags.c.pa_function, False
        ).label("pa_function"),

        func.coalesce(
            cpc_flags.c.specific_cpc, False
        ).label("specific_cpc"),

        func.coalesce(
            cpc_flags.c.agricultural_cpc, False
        ).label("agricultural_cpc"),

        func.coalesce(
            ipc_flags.c.specific_ipc, False
        ).label("specific_ipc"),

        func.coalesce(
            ipc_flags.c.agricultural_ipc, False
        ).label("agricultural_ipc"),
    )
    .select_from(family_scope)
    .outerjoin(
        keyword_flags,
        family_scope.c.fam_id
        == keyword_flags.c.fam_id
    )
    .outerjoin(
        cpc_flags,
        family_scope.c.fam_id
        == cpc_flags.c.fam_id
    )
    .outerjoin(
        ipc_flags,
        family_scope.c.fam_id
        == ipc_flags.c.fam_id
    )
    .subquery()
)

print("Combined family-level evidence prepared.")


Combined family-level evidence prepared.


## Three-route selection

R1 and R2 are direct; R3 requires agricultural CPC/IPC **and** an enabling-technology term **and** a PA-function term. Existing classification exclusions constrain R3, while topical exclusions constrain all routes except protected `A01B79/005` families.

In [12]:
# THREE ROUTES — same family-level SQL architecture as Step0_PA_v9
R1 = or_(flags.c.specific_cpc, flags.c.specific_ipc)
R2 = flags.c.specific_phrase

agricultural_class = or_(flags.c.agricultural_cpc, flags.c.agricultural_ipc)

# Unified R3: agricultural classification + enabling technology + PA function.
r3_base = and_(
    agricultural_class,
    flags.c.enabling_technology,
    flags.c.pa_function,
)

# Preserve existing CPC/IPC exclusions only for the technological route.
if excluded_families is not None:
    R3 = and_(
        r3_base,
        ~flags.c.fam_id.in_(db.query(excluded_families.c.fam_id)),
    )
else:
    R3 = r3_base

# Thematic title exclusions apply to all routes, but never exclude A01B79/005.
if topical_exclusions is not None:
    topical_allowed = or_(
        ~flags.c.fam_id.in_(db.query(topical_exclusions.c.fam_id)),
        flags.c.fam_id.in_(db.query(protected_families.c.fam_id)),
    )
else:
    topical_allowed = True

sector_query = (
    db.query(
        *flags.c,
        R1.label("route_1"),
        R2.label("route_2"),
        R3.label("route_3"),
    )
    .select_from(flags)
    .filter(and_(or_(R1, R2, R3), topical_allowed))
)
selected_families = sector_query.subquery()
print("Final family-level query: R1 OR R2 OR R3")

Final family-level query: R1 OR R2 OR R3


In [13]:
# Count UNIQUE DOCDB families for each route combination
route_counts_query = (
    db.query(
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
        func.count(func.distinct(selected_families.c.fam_id)).label("n_families"),
    )
    .select_from(selected_families)
    .group_by(
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
    )
    .order_by(func.count(func.distinct(selected_families.c.fam_id)).desc())
)
route_counts = patstat.df(route_counts_query)
if route_counts.empty:
    raise ValueError("No patent families retrieved")
total_families = int(route_counts["n_families"].sum())
route_counts["share_pct"] = (100 * route_counts["n_families"] / total_families).round(1)
print(f"Total DOCDB families: {total_families:,}")
display(route_counts)


Total DOCDB families: 5,181


,route_1,route_2,route_3,n_families,share_pct
0,True,False,False,3062,59.1
1,False,False,True,1419,27.4
2,False,True,False,305,5.9
3,True,False,True,287,5.5
4,False,True,True,52,1.0
5,True,True,False,48,0.9
6,True,True,True,8,0.2


In [14]:
# Route totals and incremental contributions (overlaps are expected)
for route in ["route_1", "route_2", "route_3"]:
    n = int(route_counts.loc[route_counts[route] == True, "n_families"].sum())
    exclusive = int(route_counts.loc[
        (route_counts[route] == True) &
        route_counts[[r for r in ["route_1", "route_2", "route_3"] if r != route]].eq(False).all(axis=1),
        "n_families"
    ].sum())
    print(f"{route}: {n:,} families; route-exclusive: {exclusive:,}")


route_1: 3,405 families; route-exclusive: 3,062
route_2: 413 families; route-exclusive: 305
route_3: 1,766 families; route-exclusive: 1,419



# Manual validation

## 13. Random sampling

A simple random sample of 100 DOCDB patent families
is selected from the retrieved population.

Sampling is performed without replacement.

Each family has the same probability of selection.

A fixed random seed ensures reproducibility when
the underlying population remains unchanged.

Manual validation assesses whether the retrieved
inventions are relevant to the Precision Agriculture
sector.

The following categories are used:

- **A — Core PA:** Innovations directly implementing
  precision-agriculture functions.
- **B — Supporting PA:** Technologies directly supporting
  precision-agriculture functions.
- **C — Out of scope:** Generic agricultural machinery
  improvements or unrelated inventions.

The final binary relevance label is:

- 1 = Core or Supporting Precision Agriculture.
- 0 = Out of scope.

Retrieval-route indicators are not used as automatic
relevance labels.


In [15]:

sampling_query = (
    db.query(
        selected_families.c.fam_id
    )
    .select_from(selected_families)
    .distinct()
)

sampling_population = patstat.df(sampling_query)

if sampling_population.empty:
    raise ValueError("No families available for sampling.")

if sampling_population["fam_id"].isna().any():
    raise ValueError("Missing family identifiers.")

sampling_population["fam_id"] = (
    sampling_population["fam_id"].astype("int64")
)

sampling_population = (
    sampling_population
    .drop_duplicates("fam_id")
    .sort_values("fam_id")
    .reset_index(drop=True)
)

assert len(sampling_population) == total_families

print(
    f"Families available for sampling: "
    f"{len(sampling_population):,}"
)

display(sampling_population.head())


Families available for sampling: 5,181


,fam_id
0,3828303
1,3828568
2,3831289
3,3945251
4,7628526


In [16]:

if not 0 < SAMPLE_SIZE <= len(sampling_population):
    raise ValueError("Invalid sample size.")

validation_sample = (
    sampling_population
    .sample(
        n=SAMPLE_SIZE,
        replace=False,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

assert len(validation_sample) == SAMPLE_SIZE
assert validation_sample["fam_id"].is_unique

print(f"Sampled families: {len(validation_sample)}")
display(validation_sample.head())


Sampled families: 100


,fam_id
0,49354929
1,62631918
2,45976512
3,42710657
4,47091398


In [17]:
sample_ids = (
    validation_sample["fam_id"]
    .astype(int)
    .tolist()
)

representatives_query = (
    db.query(
        representative_apps.c.fam_id,
        representative_apps.c.appln_id,
        TLS202_APPLN_TITLE.appln_title.label("title"),
        TLS203_APPLN_ABSTR.appln_abstract.label("abstract"),
    )
    .select_from(representative_apps)
    .outerjoin(
        TLS202_APPLN_TITLE,
        and_(
            representative_apps.c.appln_id ==
                TLS202_APPLN_TITLE.appln_id,
            TLS202_APPLN_TITLE.appln_title_lg == "en"
        )
    )
    .join(
        TLS203_APPLN_ABSTR,
        and_(
            representative_apps.c.appln_id ==
                TLS203_APPLN_ABSTR.appln_id,
            TLS203_APPLN_ABSTR.appln_abstract_lg == "en"
        )
    )
    .filter(
        representative_apps.c.fam_id.in_(sample_ids)
    )
)

representatives = patstat.df(representatives_query)

print(
    "Representative applications retrieved:",
    len(representatives)
)

assert len(representatives) == SAMPLE_SIZE
assert representatives["fam_id"].is_unique

display(representatives.head())

Representative applications retrieved: 100


,fam_id,appln_id,title,abstract
0,77095170,555195094,Grain crop harvesting robot operation safety c...,According to the operation safety control devi...
1,46125370,363727319,Agricultural device,The agricultural device has a satellite naviga...
2,57909094,474800631,Weighted multi-year yield analysis for prescri...,A multi-year yield analysis in precision agric...
3,53835870,447726443,Hybrid flow nozzle and control system,Embodiments include individual physical spray ...
4,54140764,444777428,Systems and methods for sorting seedlings for ...,A germination facility includes a seedling gro...


In [18]:
# Duplicate representative-text cell removed. Continue to the next cell.


In [19]:

if representatives.empty:
    raise ValueError(
        "No representative applications retrieved."
    )

representatives["fam_id"] = (
    representatives["fam_id"].astype("int64")
)

validation_sheet = (
    validation_sample[["fam_id"]]
    .merge(
        representatives,
        on="fam_id",
        how="left",
        validate="one_to_one",
        sort=False
    )
)

if validation_sheet["appln_id"].isna().any():
    raise ValueError(
        "Some sampled families have no application."
    )

validation_sheet["appln_id"] = (
    validation_sheet["appln_id"].astype("int64")
)

validation_sheet[["title", "abstract"]] = (
    validation_sheet[["title", "abstract"]]
    .fillna("")
)

# Empty columns for manual validation
validation_sheet["category"] = ""
validation_sheet["label"] = ""
validation_sheet["reason"] = ""

assert len(validation_sheet) == SAMPLE_SIZE
assert validation_sheet["fam_id"].is_unique

print("Validation table ready:", len(validation_sheet))

display(validation_sheet.head())


Validation table ready: 100


,fam_id,appln_id,title,abstract,category,label,reason
0,49354929,416672131,Crop feeler system and method,"In some embodiments, a crop feeler system auto...",,,
1,62631918,496362859,WORK VEHICLE,PROBLEM TO BE SOLVED: To provide a work vehicl...,,,
2,45976512,352568205,MOVING GEOFENCE FOR MACHINE TRACKING IN AGRICU...,An agricultural vehicle tracking system provid...,,,
3,42710657,329936617,Trenching Device And System,A device for trenching and a system of control...,,,
4,47091398,376850045,Targeted application system for tobacco bud in...,The invention relates to a targeted applicatio...,,,


In [20]:
# Add route membership for each sampled family (one SQL row per family)
route_sample_query = (
    db.query(
        selected_families.c.fam_id,
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
    )
    .select_from(selected_families)
    .filter(selected_families.c.fam_id.in_(sample_ids))
)
route_sample = patstat.df(route_sample_query)
route_sample["fam_id"] = route_sample["fam_id"].astype("int64")
route_sample = route_sample.drop_duplicates("fam_id")
validation_sheet = validation_sheet.merge(route_sample, on="fam_id", validate="one_to_one")
assert len(validation_sheet) == SAMPLE_SIZE
print("Route flags attached to validation sheet")


Route flags attached to validation sheet


In [21]:
validation_path = OUTPUT_DIR / "hand_validation_PA_FINAL_representative.csv"
validation_sheet.to_csv(validation_path, index=False, encoding="utf-8-sig")
print("Validation CSV:", validation_path)
print("Sample size:", len(validation_sheet))


Validation CSV: /home/jovyan/team-38-PA/notebooks/outputs/step0/hand_validation_PA_FINAL_representative.csv
Sample size: 100


In [22]:
import pandas as pd
from sqlalchemy import text, bindparam

# 1. Load the validated sample
df = pd.read_csv(
    "outputs/step0/hand_validation_PA_FINAL_representative_labeled.csv"
)

# 2. Select Out of Scope patents belonging to R1
r1_c = df[
    (df["category"] == "C") &
    (df["route_1"] == True)
].copy()

print("Out of Scope families in R1:", len(r1_c))

# 3. Get the representative application IDs
appln_ids = r1_c["appln_id"].astype(int).tolist()

# 4. Check protected CPC on the representative application
sql = text("""
SELECT
    appln_id,
    MAX(
        CASE
            WHEN REPLACE(cpc_class_symbol, ' ', '') = 'A01B79/005'
            THEN 1
            ELSE 0
        END
    ) AS has_protected_cpc
FROM tls224_appln_cpc
WHERE appln_id IN :appln_ids
GROUP BY appln_id
""").bindparams(bindparam("appln_ids", expanding=True))

result = pd.DataFrame(
    db.execute(sql, {"appln_ids": appln_ids}).fetchall(),
    columns=["appln_id", "has_protected_cpc"]
)

# 5. Merge results with the validation sample
check = r1_c[
    ["fam_id", "appln_id", "title", "category"]
].merge(
    result,
    on="appln_id",
    how="left"
)

check["has_protected_cpc"] = (
    check["has_protected_cpc"]
    .fillna(0)
    .astype(int)
)

display(check.sort_values(
    "has_protected_cpc",
    ascending=False
))

print(
    "\nFamilies to reclassify:",
    check["has_protected_cpc"].sum()
)

Out of Scope families in R1: 13


,fam_id,appln_id,title,category,has_protected_cpc
0,42710657,329936617,Trenching Device And System,C,1
1,91382416,613021557,Weight transferring hitch system for agricultu...,C,1
4,73475976,541375651,WORK SUPPORT SYSTEM AND WORK SUPPORT DEVICE PO...,C,1
7,54868809,447726441,Broadband spray nozzle systems and methods,C,1
5,66061154,511205570,Compounding improvement method for rhizosphere...,C,1
12,53835870,447726443,Hybrid flow nozzle and control system,C,1
2,68764050,523332566,Automated reverse implement parking,C,0
3,74114058,544366007,DIRECTION ADJUSTING DEVICE AND APPLICATION THE...,C,0
6,60990405,488977049,Loader Positioning System,C,0
8,38884482,41513578,Method is for control of fruit harvesting mach...,C,0



Families to reclassify: 6
